# Mental Maths Quiz — Performance Analysis

This notebook analyses quiz performance across sessions, levels, and operators.

**Rolling average window:** 5 sessions for all trend lines.

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Config ──────────────────────────────────────────────────────────────────
ROLLING_WINDOW = 5
LEVELS = [1, 2, 3, 4, 5]
OPERATORS = ['addition', 'subtraction', 'multiplication', 'division', 'percentage']

COLOUR_RT  = '#2196F3'   # blue  — response time
COLOUR_ACC = '#FF9800'   # amber — accuracy

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'font.size': 10,
})

## Load & Prepare Data

In [ ]:
questions = pd.read_csv('questions.csv')
sessions  = pd.read_csv('sessions.csv')

# ── Parse dates & sort ───────────────────────────────────────────────────────
for df in [questions, sessions]:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

questions.sort_values('date', inplace=True)
sessions.sort_values('date',  inplace=True)

# ── Normalise dtypes ─────────────────────────────────────────────────────────
questions['is_correct']      = questions['is_correct'].astype(bool)
questions['response_time_ms'] = pd.to_numeric(questions['response_time_ms'], errors='coerce')
questions['difficulty_level'] = pd.to_numeric(questions['difficulty_level'], errors='coerce')
questions['operator']        = questions['operator'].str.strip().str.lower()
sessions['level']            = pd.to_numeric(sessions['level'], errors='coerce')

# ── Merge level onto questions (via session_id) ──────────────────────────────
questions = questions.merge(
    sessions[['session_id', 'level']].rename(columns={'level': 'session_level'}),
    on='session_id', how='left'
)

print(f"Questions: {len(questions):,} rows")
print(f"Sessions:  {len(sessions):,} rows")
print(f"Levels present:    {sorted(sessions['level'].dropna().unique().tolist())}")
print(f"Operators present: {sorted(questions['operator'].dropna().unique().tolist())}")

---
## Graph 1 — Level Comparison (static snapshot)

Single chart: average response time (left axis, blue) and accuracy rate (right axis, amber) per level.

In [ ]:
level_stats = (
    questions
    .groupby('session_level')
    .agg(
        avg_rt   = ('response_time_ms', 'mean'),
        accuracy = ('is_correct',       'mean'),
    )
    .reindex(LEVELS)
    .reset_index()
    .rename(columns={'session_level': 'level'})
)

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax2 = ax1.twinx()

x = level_stats['level']

bars = ax1.bar(x - 0.18, level_stats['avg_rt'], width=0.35,
               color=COLOUR_RT, alpha=0.85, label='Avg Response Time (ms)')
line = ax2.plot(x, level_stats['accuracy'] * 100, 'o-',
                color=COLOUR_ACC, linewidth=2.2, markersize=7,
                label='Accuracy (%)')

ax1.set_xlabel('Level')
ax1.set_ylabel('Avg Response Time (ms)', color=COLOUR_RT)
ax2.set_ylabel('Accuracy (%)',           color=COLOUR_ACC)
ax1.tick_params(axis='y', labelcolor=COLOUR_RT)
ax2.tick_params(axis='y', labelcolor=COLOUR_ACC)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax2.set_ylim(0, 110)
ax1.set_xticks(LEVELS)
ax1.set_xlim(0.5, 5.5)
ax1.spines['right'].set_visible(False)

# Combined legend
handles = [bars, line[0]]
ax1.legend(handles, [h.get_label() for h in handles],
           loc='upper right', framealpha=0.8)

plt.title('Level Comparison — Avg Response Time & Accuracy', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('graph1_level_comparison.png', bbox_inches='tight')
plt.show()

---
## Graph 2 — Development Overview (one graph per level)

5 graphs. X axis = cumulative session number at that level. Dual Y: rolling-5 avg response time (blue) and rolling-5 accuracy (amber).

In [ ]:
def build_session_series(df_q, level=None, operator=None):
    """Aggregate questions → per-session stats, optionally filtered."""
    mask = pd.Series(True, index=df_q.index)
    if level    is not None: mask &= df_q['session_level'] == level
    if operator is not None: mask &= df_q['operator']       == operator

    grp = (
        df_q[mask]
        .groupby('session_id')
        .agg(
            date     = ('date', 'min'),
            avg_rt   = ('response_time_ms', 'mean'),
            accuracy = ('is_correct',       'mean'),
        )
        .reset_index()
        .sort_values('date')
        .reset_index(drop=True)
    )
    grp['session_num'] = range(1, len(grp) + 1)
    grp['roll_rt']  = grp['avg_rt'].rolling(ROLLING_WINDOW, min_periods=1).mean()
    grp['roll_acc'] = grp['accuracy'].rolling(ROLLING_WINDOW, min_periods=1).mean() * 100
    return grp


def plot_dual_trend(ax1, series, title):
    """Draw rolling RT + accuracy on a dual-axis chart."""
    if series.empty:
        ax1.text(0.5, 0.5, 'No data', transform=ax1.transAxes,
                 ha='center', va='center', color='grey')
        ax1.set_title(title); return

    ax2 = ax1.twinx()
    x   = series['session_num']

    ax1.plot(x, series['avg_rt'],  color=COLOUR_RT,  alpha=0.25, linewidth=1)
    ax1.plot(x, series['roll_rt'], color=COLOUR_RT,  linewidth=2, label='Resp Time (roll avg)')

    ax2.plot(x, series['accuracy'] * 100, color=COLOUR_ACC, alpha=0.25, linewidth=1)
    ax2.plot(x, series['roll_acc'],        color=COLOUR_ACC, linewidth=2, label='Accuracy (roll avg)')

    ax1.set_ylabel('Response Time (ms)', color=COLOUR_RT)
    ax2.set_ylabel('Accuracy (%)',        color=COLOUR_ACC)
    ax1.tick_params(axis='y', labelcolor=COLOUR_RT)
    ax2.tick_params(axis='y', labelcolor=COLOUR_ACC)
    ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
    ax2.set_ylim(0, 110)

    if len(x) > 1:
        ax1.set_xlim(1, x.max())

    ax1.set_xlabel('Cumulative Session Number')
    ax1.spines['top'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax1.set_title(title, fontsize=11)

    # Combined legend
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8, framealpha=0.7)


# ── Render ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(5, 1, figsize=(10, 28))
fig.suptitle('Development Overview — Rolling Avg per Level', fontsize=14, y=1.005)

for ax, lvl in zip(axes, LEVELS):
    series = build_session_series(questions, level=lvl)
    plot_dual_trend(ax, series, f'Level {lvl}')

plt.tight_layout(h_pad=3)
plt.savefig('graph2_development_overview.png', bbox_inches='tight')
plt.show()

---
## Graph 3 — Operator Development (5 operators × 5 levels = 25 graphs)

Arranged as a 5×5 grid (rows = levels 1-5, columns = operators). Same dual-axis rolling format as Graph 2.

In [ ]:
op_labels = {
    'addition':       'Addition',
    'subtraction':    'Subtraction',
    'multiplication': 'Multiplication',
    'division':       'Division',
    'percentage':     'Percentage',
}

fig, axes = plt.subplots(
    nrows=len(LEVELS),
    ncols=len(OPERATORS),
    figsize=(22, 26),
    constrained_layout=True,
)
fig.suptitle('Operator Development — Rolling Avg per Operator × Level',
             fontsize=15, y=1.01)

# Column headers
for col, op in enumerate(OPERATORS):
    axes[0, col].set_title(op_labels[op], fontsize=11, fontweight='bold', pad=14)

for row, lvl in enumerate(LEVELS):
    for col, op in enumerate(OPERATORS):
        ax = axes[row, col]

        # Row label on the far-left column
        if col == 0:
            ax.set_ylabel(f'Level {lvl}\n\nResp Time (ms)', color=COLOUR_RT, fontsize=9)

        series = build_session_series(questions, level=lvl, operator=op)

        if series.empty:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                    ha='center', va='center', color='grey', fontsize=9)
            ax.spines['top'].set_visible(False)
            continue

        ax2 = ax.twinx()
        x   = series['session_num']

        ax.plot(x, series['avg_rt'],  color=COLOUR_RT,  alpha=0.2, linewidth=0.8)
        ax.plot(x, series['roll_rt'], color=COLOUR_RT,  linewidth=1.6)

        ax2.plot(x, series['accuracy'] * 100, color=COLOUR_ACC, alpha=0.2, linewidth=0.8)
        ax2.plot(x, series['roll_acc'],        color=COLOUR_ACC, linewidth=1.6)

        ax.tick_params(axis='y', labelcolor=COLOUR_RT, labelsize=7)
        ax2.tick_params(axis='y', labelcolor=COLOUR_ACC, labelsize=7)
        ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
        ax2.set_ylim(0, 110)

        ax.set_xlabel('Session #', fontsize=7)
        if len(x) > 1:
            ax.set_xlim(1, x.max())

        ax.spines['top'].set_visible(False)
        ax2.spines['top'].set_visible(False)

        # Only show right-axis label on the last column
        if col == len(OPERATORS) - 1:
            ax2.set_ylabel('Accuracy (%)', color=COLOUR_ACC, fontsize=8)
        else:
            ax2.set_yticklabels([])

# Shared legend at the bottom
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=COLOUR_RT,  linewidth=2, label='Response Time (rolling avg)'),
    Line2D([0], [0], color=COLOUR_ACC, linewidth=2, label='Accuracy (rolling avg)'),
    Line2D([0], [0], color=COLOUR_RT,  linewidth=1, alpha=0.3, label='Response Time (raw)'),
    Line2D([0], [0], color=COLOUR_ACC, linewidth=1, alpha=0.3, label='Accuracy (raw)'),
]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.015))

plt.savefig('graph3_operator_development.png', bbox_inches='tight')
plt.show()

---
*All charts saved as PNG files alongside this notebook.*